In [1]:
# Fix for common Windows compatibility issues
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Debugging import issues: show interpreter, cwd, sys.path, and locate module file
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# The module lives one level up from this notebook; add that parent explicitly
module_parent = str(Path.cwd().parent)
if module_parent not in sys.path:
    sys.path.insert(0, module_parent)
    print('Inserted', module_parent, 'into sys.path')
else:
    print('Parent already on sys.path:', module_parent)

# Add the helper functions directory to the path
helper_functions_path = Path(module_parent) / 'helperFunctions' / 'StimulusModeling'
if str(helper_functions_path) not in sys.path:
    sys.path.insert(0, str(helper_functions_path))

# Now import the module (will import from path printed above)
import stim_transformations as stf

Inserted /mnt/DataDrive3/emeyer/TreeShrewObjectRecognition into sys.path


In [2]:
projectpath = '/mnt/DataDrive3/emeyer/TreeShrewObjectRecognition/'
taskName = 'Camel_v2_test_nn' #'Camel_v2_test_nn' 'Camel_Rhino_test_nn' , 'Camel_v2_test_nn', 'Camel_background_matrix'
img_size = 10
imgPath = f'{projectpath}stimulusSets/isettreeshrew/treeshrew_{img_size}/{taskName}_quad/'

In [4]:
from scipy import stats
from skimage.morphology import remove_small_holes

## Create texture metamers from isetbio filtered images
# Define save path for transformations
savePath = f'{projectpath}stimulusSets/{taskName}/binary_object_scaled/'
loadPath = f'{projectpath}stimulusSets/{taskName}/register_scaled/'

# Get list of image filenames
img_filenames = [f for f in os.listdir(loadPath) if f.endswith('.png')]
for img_filename in img_filenames:
    # Load the original image and convert to grayscale
    img = cv2.imread(os.path.join(loadPath, img_filename), cv2.IMREAD_GRAYSCALE)

    # Threshold the image to create a binary image and fill small holes
    backgroundVal = stats.mode(img.flatten())[0]
    binary = img != backgroundVal
    filled_mask = remove_small_holes(binary, area_threshold=3) != True

    # plt.figure()
    # plt.imshow(filled_mask, cmap='gray')
    # plt.show()

    # Save the transformed image
    plt.imsave(os.path.join(savePath, img_filename), filled_mask, cmap='gray', vmin=0, vmax=1)

In [5]:
## Create texture metamers from isetbio filtered images
# Define save path for transformations
savePath = f'{imgPath}texture_inplace/'
loadPath = f'{imgPath}original/merged/'

# Get list of image filenames
img_filenames = [f for f in os.listdir(loadPath) if f.endswith('.png')]
for img_filename in img_filenames:
    # Load the original image and convert to grayscale
    img = cv2.imread(os.path.join(loadPath, img_filename), cv2.IMREAD_GRAYSCALE)
    
    # Apply texture metamer transformation
    texture_img = stf.transform_image(img, operation='texture_inplace')

    # Save the transformed image
    plt.imsave(os.path.join(savePath, img_filename), texture_img, cmap='gray', vmin=0, vmax=255)

Running on GPU!
Final crop: x=101, y=59, size=112x116, target=112x116
Image shape after processing: torch.Size([1, 1, 112, 116])


/home/arcarolab_adm/anaconda3/lib/python3.11/site-packages/plenoptic/tools/validate.py:343: UserWarning: Validating whether model can work with coarse-to-fine synthesis -- this can take a while!
  warnings.warn(


  0%|          | 0/1500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
## Create texture metamers from isetbio filtered images
# Define save path for transformations
savePath = f'{imgPath}texture_crop/'
loadPath = f'{imgPath}original/merged/'

# Get list of image filenames
img_filenames = [f for f in os.listdir(loadPath) if f.endswith('.png')]
for img_filename in img_filenames:
    # Load the original image and convert to grayscale
    img = cv2.imread(os.path.join(loadPath, img_filename), cv2.IMREAD_GRAYSCALE)
    
    # Apply texture metamer transformation
    _, texture_img = stf.transform_image(img, operation='texture_crop')

    # Save the transformed image
    plt.imsave(os.path.join(savePath, img_filename), texture_img, cmap='gray', vmin=0, vmax=255)

Running on GPU!


  0%|          | 0/1500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
## Create skeletonized object from isetbio filtered images
# Define save path for transformations
savePath = f'{imgPath}skeleton_scaled/'
loadPath = f'{imgPath}register_scaled/merged/'

# Get list of image filenames
img_filenames = [f for f in os.listdir(loadPath) if f.endswith('.png')]
for img_filename in img_filenames:
    # Load the original image and convert to grayscale
    img = cv2.imread(os.path.join(loadPath, img_filename), cv2.IMREAD_GRAYSCALE)
    
    # Apply texture metamer transformation
    binary, skeleton_img = stf.transform_image(img, operation='skeleton')

    # Save the transformed image
    plt.imsave(os.path.join(savePath, img_filename), skeleton_img, cmap='gray', vmin=0, vmax=255)

In [10]:
import numpy as np
import pandas as pd
from scipy.spatial import distance_matrix

if taskName == 'Camel_Rhino_test_nn':
    distractor = 'rhino'
    dist_idx = [0, 3, 11, 16, 70, 76, 92]
else:
    distractor = 'wrench'
    dist_idx = [0, 14, 26, 83, 87, 92, 93]
targ_idx = [0,  6, 9, 13, 17, 32, 38, 40, 44, 55, 60, 65]

loadPath = f'{imgPath}original/merged/'
network = 'alexnet_mouse'  # 'alexnet_mouse', 'alexnet', 'vgg16'
act_size = 4096

act_fc3_targ = np.zeros((len(targ_idx), act_size))
for idx, tt in enumerate(targ_idx):
    # Load the original image and convert to grayscale
    img = cv2.imread(f'{loadPath}camel_{tt}.png', cv2.IMREAD_GRAYSCALE)

    act,lay = stf.transform_image(img, operation='NN', layer_types = ['Linear'], network=network)
    print(lay[-1])
    act_fc3_targ[idx,:] = act[-1][0]

act_fc3_dist = np.zeros((len(dist_idx), act_size))
for idx, dd in enumerate(dist_idx):
    # Load the original image and convert to grayscale
    img = cv2.imread(f'{loadPath}{distractor}_{dd}.png', cv2.IMREAD_GRAYSCALE)

    act,_ = stf.transform_image(img, operation='NN', layer_types = ['Linear'], network=network)
    act_fc3_dist[idx,:] = act[-1][0]

# Compute pairwise distances between target and distractor activations
distances = distance_matrix(act_fc3_targ, act_fc3_dist)
distances_df = pd.DataFrame(distances.flatten())
distances_df.to_csv(f'./distData/{taskName}/{network}_distances_linear.csv', index=False, header=False)
# np.savetxt(f'./distData/{taskName}/{network}_distances.csv', distances)

Loading alexnet_bn_ir_64x64_input_pool_6. Pretrained: True. Model Family: imagenet.
Loaded parameters from /mnt/DataDrive3/emeyer/TreeShrewObjectRecognition/helperFunctions/StimulusModeling/mouse_vision/model_ckpts/alexnet_bn_ir.pt
Linear: classifier.5
Loading alexnet_bn_ir_64x64_input_pool_6. Pretrained: True. Model Family: imagenet.
Loaded parameters from /mnt/DataDrive3/emeyer/TreeShrewObjectRecognition/helperFunctions/StimulusModeling/mouse_vision/model_ckpts/alexnet_bn_ir.pt
Linear: classifier.5
Loading alexnet_bn_ir_64x64_input_pool_6. Pretrained: True. Model Family: imagenet.
Loaded parameters from /mnt/DataDrive3/emeyer/TreeShrewObjectRecognition/helperFunctions/StimulusModeling/mouse_vision/model_ckpts/alexnet_bn_ir.pt
Linear: classifier.5
Loading alexnet_bn_ir_64x64_input_pool_6. Pretrained: True. Model Family: imagenet.
Loaded parameters from /mnt/DataDrive3/emeyer/TreeShrewObjectRecognition/helperFunctions/StimulusModeling/mouse_vision/model_ckpts/alexnet_bn_ir.pt
Linear: c